# Euler's Method for Ordinary Differential Equations

## The Problem

Given a first-order initial value problem (IVP):

$$\frac{dy}{dt} = f(t, y), \qquad y(t_0) = y_0$$

find an approximation to $y(t)$ on some interval $[t_0, T]$. The function $f$ tells us the
**slope** of the solution at any point $(t, y)$ — but not the solution itself.

---

## The Key Idea: Follow the Tangent

Taylor-expand the true solution $y(t)$ around $t_k$:

$$y(t_k + h) = y(t_k) + h\,y'(t_k) + \frac{h^2}{2}y''(t_k) + O(h^3)$$

Since $y'(t_k) = f(t_k, y(t_k))$, truncating after the linear term gives:

$$y(t_k + h) \approx y(t_k) + h\,f(t_k, y(t_k))$$

This is **Euler's method**: take one step in the direction of the current slope and discard all higher-order terms.

---

## The Algorithm

Choose a step size $h > 0$ and define $t_k = t_0 + kh$ for $k = 0, 1, 2, \dots, N$ where $N = (T - t_0)/h$.

**Initialize:** $y_0 = y(t_0)$ (given).

**Iterate** for $k = 0, 1, \dots, N-1$:

$$\boxed{y_{k+1} = y_k + h\,f(t_k,\, y_k)}$$

Each step requires exactly **one evaluation** of $f$.

---

## Local Truncation Error

The **local truncation error** (LTE) is the error made in a single step, assuming the previous value $y_k$ is exact:

$$\tau_{k+1} = y(t_{k+1}) - y(t_k) - h\,f(t_k, y(t_k))$$

From the Taylor expansion:

$$y(t_{k+1}) = y(t_k) + h\,f(t_k, y(t_k)) + \frac{h^2}{2}y''(t_k) + O(h^3)$$

So:

$$\tau_{k+1} = \frac{h^2}{2}y''(\xi_k) + O(h^3) = O(h^2), \quad \xi_k \in (t_k, t_{k+1})$$

The LTE is **second order** in $h$ — the first discarded Taylor term dominates.

---

## Global Truncation Error

The **global error** at time $T = t_0 + Nh$ accumulates $N = (T-t_0)/h$ local errors:

$$E_N = y(T) - y_N$$

Each step contributes an LTE of $O(h^2)$, and there are $O(1/h)$ steps, so naively:

$$E_N \sim N \cdot O(h^2) = \frac{T - t_0}{h} \cdot O(h^2) = O(h)$$

More precisely, under a Lipschitz condition $|f(t,u) - f(t,v)| \leq L|u - v|$:

$$\boxed{|E_N| \leq \frac{h\,\|y''\|_\infty}{2L}\left(e^{L(T-t_0)} - 1\right) = O(h)}$$

Euler's method is **first-order**: halving $h$ halves the global error. This makes it the ODE analogue of the left-endpoint rectangle rule for integration.

---

## Geometric Interpretation

At each step, Euler's method constructs the **tangent line** to the true solution at $(t_k, y_k)$ and walks along it for time $h$. The error arises because the true solution curves away from its tangent. The faster $y$ curves (large $|y''|$), the more error per step.

The curvature of $y$ is:

$$y'' = \frac{d}{dt}f(t,y) = f_t + f_y \cdot f$$

So the LTE can be written as:

$$\tau_{k+1} = \frac{h^2}{2}(f_t + f_y f)\big|_{(t_k, y_k)} + O(h^3)$$

This gives a computable estimate of where the method struggles: regions where $f$ changes rapidly in $t$ or $y$.

---

## Stability

Euler's method is not just about accuracy — it can **blow up** even on stable ODEs if $h$ is too large.

Test the method on the model problem $y' = \lambda y$ with $\lambda < 0$ (a decaying solution). One step gives:

$$y_{k+1} = y_k + h\lambda y_k = (1 + h\lambda)\,y_k$$

After $N$ steps: $y_N = (1 + h\lambda)^N y_0$.

The true solution decays: $y(T) = e^{\lambda T} y_0 \to 0$.

For the numerical solution to also decay, we need:

$$|1 + h\lambda| < 1$$

For real $\lambda < 0$, this requires:

$$h < \frac{2}{|\lambda|}$$

This is the **stability condition** for Euler's method. If violated, the numerical solution grows without bound even though the true solution decays — a catastrophic failure.

The **stability region** of Euler's method in the complex $h\lambda$-plane is:

$$\mathcal{S} = \{z \in \mathbb{C} : |1 + z| < 1\}$$

a disk of radius 1 centred at $z = -1$. Stiff problems (with large $|\lambda|$) force $h$ to be tiny for stability, making Euler's method impractical.

---

## Euler's Method as a Quadrature Rule

Rearranging the IVP:

$$y(T) = y_0 + \int_{t_0}^T f(t, y(t))\,dt$$

Euler's method approximates this integral using the **left-endpoint rectangle rule** on each subinterval:

$$\int_{t_k}^{t_{k+1}} f(t, y(t))\,dt \approx h\,f(t_k, y_k)$$

The connection to earlier notes is exact:

| ODE method | Quadrature analogue | Global order |
|---|---|---|
| Euler (explicit) | Left-endpoint rule | $O(h)$ |
| Trapezoidal method | Trapezoid rule | $O(h^2)$ |
| Midpoint method | Midpoint rule | $O(h^2)$ |
| RK4 | Simpson / Gauss | $O(h^4)$ |

Every improvement in quadrature has a direct ODE analogue.

---

## Variants

**Implicit (Backward) Euler:**

$$y_{k+1} = y_k + h\,f(t_{k+1},\, y_{k+1})$$

Uses the slope at the **next** point. Requires solving an equation for $y_{k+1}$ at each step (nonlinear solve if $f$ is nonlinear), but has an **unconditionally stable** stability region $\{z : |1/(1-z)| < 1\}$ — the entire left half-plane. Ideal for stiff problems.

**Symplectic Euler** (for Hamiltonian systems):

$$p_{k+1} = p_k - h\,\nabla_q H(q_k, p_k), \qquad q_{k+1} = q_k + h\,\nabla_p H(q_k, p_{k+1})$$

Preserves a modified Hamiltonian exactly — no energy drift over long integrations.

---

## Conditions and Caveats

| Concern | Detail |
|---|---|
| Only first-order accurate | Use RK4 or higher for practical work |
| Stability restriction | $h < 2/|\lambda|$ for stiff problems — can force extremely small steps |
| Error accumulation | Global error grows exponentially with integration time for unstable systems |
| Systems of ODEs | Apply componentwise: $\mathbf{y}_{k+1} = \mathbf{y}_k + h\,\mathbf{f}(t_k, \mathbf{y}_k)$ |
| Higher-order ODEs | Reduce to first-order system first (e.g. $y'' = g$ becomes $[y, v]' = [v, g]$) |

In [1]:
def euler_method_ode(func, y0, t0, tf, dt):
    """
    Solves an ordinary differential equation (ODE) using the Euler method.

    Parameters:
    func: A function that takes two arguments (t, y) and returns the derivative dy/dt.
    y0: The initial value of y at time t0.
    t0: The initial time.
    tf: The final time.
    dt: The time step size.

    Returns:
    A list of tuples (t, y) representing the time and corresponding value of y at each time step.
    """
    times = []
    values = []
    
    t = t0
    y = y0
    
    while t <= tf:
        times.append(t)
        values.append(y)
        
        # Compute the derivative using the provided function
        dydt = func(t, y)
        
        # Update y using the Euler method
        y += dydt * dt
        
        # Increment time by dt
        t += dt
    
    return list(zip(times, values))

# Modified Euler Method (Heun’s Method)

## The Problem

Given the IVP:

$$
\frac{dy}{dt} = f(t, y), \qquad y(t_0) = y_0
$$

we seek a more accurate alternative to Euler’s method without greatly increasing computational cost.

---

## The Key Idea: Correct the Slope

Euler’s method uses only the slope at the start of the interval:

$$
y_{k+1} = y_k + h,f(t_k, y_k)
$$

This ignores how the slope changes across the step.

Modified Euler improves this by:

1. Predicting where Euler would land
2. Recomputing the slope there
3. Averaging the two slopes

---

## Derivation (Predictor–Corrector View)

**Step 1: Predictor (Euler step)**

$$
y_{k+1}^{(p)} = y_k + h,f(t_k, y_k)
$$

**Step 2: Corrector (average slope)**

$$
y_{k+1} = y_k + \frac{h}{2}\left[f(t_k, y_k) + f(t_{k+1}, y_{k+1}^{(p)})\right]
$$

---

## Interpretation as a Quadrature Rule

Recall:

$$
y(t_{k+1}) = y(t_k) + \int_{t_k}^{t_{k+1}} f(t, y(t)),dt
$$

Modified Euler corresponds to the trapezoidal rule:

$$
\int_{t_k}^{t_{k+1}} f(t, y(t)),dt \approx \frac{h}{2}\left[f(t_k, y_k) + f(t_{k+1}, y_{k+1}^{(p)})\right]
$$

---

## The Algorithm

Given step size $h$:

**Initialize:** $y_0 = y(t_0)$

For $k = 0, 1, \dots, N-1$:

1. Predictor:
   $$
   y_{k+1}^{(p)} = y_k + h,f(t_k, y_k)
   $$

2. Corrector:
   $$
   y_{k+1} = y_k + \frac{h}{2}\left[f(t_k, y_k) + f(t_{k+1}, y_{k+1}^{(p)})\right]
   $$

---

## Local Truncation Error

Expand the true solution:

$$
y(t_{k+1}) = y(t_k) + h y'(t_k) + \frac{h^2}{2}y''(t_k) + O(h^3)
$$

Expand the second slope:

$$
f(t_{k+1}, y(t_{k+1})) = f(t_k, y(t_k)) + h(f_t + f_y f) + O(h^2)
$$

Averaging slopes:

$$
f(t_k, y_k) + \frac{h}{2}(f_t + f_y f) + O(h^2)
$$

Multiplying by $h$:

$$
h f(t_k, y_k) + \frac{h^2}{2}(f_t + f_y f) + O(h^3)
$$

So:

$$
\tau_{k+1} = O(h^3)
$$

---

## Global Error

Over $N \sim 1/h$ steps:

$$
E_N = O(h^2)
$$

Modified Euler is a **second-order method**.

---

## Stability

Apply to:

$$
y' = \lambda y
$$

Predictor:

$$
y_{k+1}^{(p)} = (1 + h\lambda)y_k
$$

Corrector:

$$
y_{k+1}
= y_k + \frac{h}{2}\left[\lambda y_k + \lambda(1 + h\lambda)y_k\right]
$$

So:

$$
y_{k+1}
= \left(1 + h\lambda + \frac{h^2 \lambda^2}{2}\right)y_k
$$

Amplification factor:

$$
R(z) = 1 + z + \frac{z^2}{2}, \quad z = h\lambda
$$

Stability condition:

$$
|R(z)| < 1
$$

---

## Summary

| Property             | Value               |
| -------------------- | ------------------- |
| Order                | 2                   |
| Function evaluations | 2 per step          |
| LTE                  | $O(h^3)$            |
| Global error         | $O(h^2)$            |
| Type                 | Explicit RK2 (Heun) |


In [2]:
def modified_euler_method_ode(func, y0, t0, tf, dt):
    """
    Solves an ordinary differential equation (ODE) using the Modified Euler method (Heun's method).

    Parameters:
    func: A function that takes two arguments (t, y) and returns the derivative dy/dt.
    y0: The initial value of y at time t0.
    t0: The initial time.
    tf: The final time.
    dt: The time step size.

    Returns:
    A list of tuples (t, y) representing the time and corresponding value of y at each time step.
    """
    times = []
    values = []
    
    t = t0
    y = y0
    
    while t <= tf:
        times.append(t)
        values.append(y)
        
        # Compute the derivative at the current point
        dydt = func(t, y)
        
        # Predict the value of y at the next time step using Euler's method
        y_predict = y + dydt * dt
        
        # Compute the derivative at the predicted point
        dydt_predict = func(t + dt, y_predict)
        
        # Update y using the average of the two derivatives
        y += (dydt + dydt_predict) / 2 * dt
        
        # Increment time by dt
        t += dt
    
    return list(zip(times, values))

# Second-Order Runge–Kutta Methods (RK2 Family)

## The Problem

Given the IVP:

$$
\frac{dy}{dt} = f(t, y), \qquad y(t_0) = y_0
$$

we seek a systematic improvement over Euler’s method with **second-order accuracy**.

---

## The Key Idea: Two-Point Slope Approximation

Instead of using a single slope (Euler), we approximate the average slope over the interval using **two evaluations**:

* one at the beginning,
* one at an intermediate point.

---

## General Form of RK2

Let:

$$
k_1 = f(t_k, y_k)
$$

$$
k_2 = f(t_k + \alpha h,; y_k + \alpha h k_1)
$$

Then define:

$$
\boxed{
y_{k+1} = y_k + h\left[(1 - \beta)k_1 + \beta k_2\right]
}
$$

for parameters $\alpha, \beta$ to be determined.

---

## Order Conditions (Derivation)

Expand the true solution:

$$
y(t_k + h) = y(t_k) + h y'(t_k) + \frac{h^2}{2}y''(t_k) + O(h^3)
$$

Now expand $k_2$:

$$
k_2 = f + \alpha h(f_t + f_y f) + O(h^2)
$$

Substitute into the method:

$$
y_{k+1}
= y_k + h\left[(1-\beta)f + \beta\left(f + \alpha h(f_t + f_y f)\right)\right] + O(h^3)
$$

$$
= y_k + h f + \alpha \beta h^2 (f_t + f_y f) + O(h^3)
$$

Match with the Taylor expansion:

$$
\frac{h^2}{2}(f_t + f_y f)
$$

So we require:

$$
\boxed{
\alpha \beta = \frac{1}{2}
}
$$

---

## Family of RK2 Methods

Any choice of $(\alpha, \beta)$ satisfying:

$$
\alpha \beta = \frac{1}{2}
$$

gives a **second-order method**.

---

## Common Choices

### Heun’s Method (Modified Euler)

$$
\alpha = 1, \quad \beta = \frac{1}{2}
$$

$$
y_{k+1} = y_k + \frac{h}{2}(k_1 + k_2)
$$

---

### Midpoint Method

$$
\alpha = \frac{1}{2}, \quad \beta = 1
$$

$$
y_{k+1} = y_k + h,f\left(t_k + \frac{h}{2},; y_k + \frac{h}{2}k_1\right)
$$

---

### Ralston’s Method (Minimum Error Constant)

$$
\alpha = \frac{2}{3}, \quad \beta = \frac{3}{4}
$$

Minimizes the leading error coefficient.

---

## Local and Global Error

All RK2 methods satisfy:

$$
\tau_{k+1} = O(h^3), \qquad E_N = O(h^2)
$$

They are **second-order accurate**.

---

## Interpretation

RK2 methods approximate:

$$
\int_{t_k}^{t_{k+1}} f(t, y(t)),dt
$$

using a **two-point quadrature rule along the solution path**.

Different parameter choices correspond to different quadrature approximations.

---

## Summary

| Property             | Value                   |
| -------------------- | ----------------------- |
| Stages               | 2                       |
| Order                | 2                       |
| Function evaluations | 2 per step              |
| Condition            | $\alpha \beta = 1/2$    |
| Examples             | Heun, Midpoint, Ralston |


# Fourth Order Runge–Kutta Method (Classical RK4)

## The Problem

Given a first-order initial value problem (IVP):

$$
\frac{dy}{dt} = f(t, y), \qquad y(t_0) = y_0
$$

we seek a high-accuracy method that remains explicit and avoids computing higher derivatives.

---

## The Key Idea: Sample the Slope Within the Interval

Euler’s method uses a single slope at $t_k$. RK4 improves this by sampling slopes at multiple points within $[t_k, t_{k+1}]$ and taking a weighted average.

Instead of approximating:

$$
\int_{t_k}^{t_{k+1}} f(t, y(t)),dt
$$

with a simple rule, RK4 builds a **four-point quadrature approximation along the solution trajectory**.

---

## The Construction

Define four slope evaluations:

$$
k_1 = f(t_k, y_k)
$$

$$
k_2 = f\left(t_k + \frac{h}{2},; y_k + \frac{h}{2}k_1\right)
$$

$$
k_3 = f\left(t_k + \frac{h}{2},; y_k + \frac{h}{2}k_2\right)
$$

$$
k_4 = f\left(t_k + h,; y_k + h k_3\right)
$$

These correspond to:

* $k_1$: slope at the beginning
* $k_2, k_3$: slopes at midpoint (refined)
* $k_4$: slope at the end

---

## The Algorithm

The RK4 update is:

$$
\boxed{
y_{k+1} = y_k + \frac{h}{6}\left(k_1 + 2k_2 + 2k_3 + k_4\right)
}
$$

Each step requires **four evaluations** of $f$.

---

## Derivation (Taylor Matching Idea)

Expand the true solution:

$$
y(t_k + h) = y(t_k) + h y'(t_k) + \frac{h^2}{2}y''(t_k) + \frac{h^3}{6}y'''(t_k) + \frac{h^4}{24}y^{(4)}(t_k) + O(h^5)
$$

RK4 chooses coefficients so that the weighted combination of $k_1, k_2, k_3, k_4$ matches this expansion up to $O(h^4)$.

This eliminates all terms through fourth order, leaving:

$$
\tau_{k+1} = O(h^5)
$$

---

## Local Truncation Error

The local truncation error satisfies:

$$
\tau_{k+1} = O(h^5)
$$

This comes from matching the Taylor expansion up to fourth order.

---

## Global Error

Over $N \sim 1/h$ steps:

$$
E_N = O(h^4)
$$

RK4 is a **fourth-order method**.

Halving $h$ reduces the error by roughly a factor of $16$.

---

## Geometric Interpretation

RK4 approximates the solution by combining:

* the initial slope ($k_1$),
* two midpoint estimates ($k_2, k_3$),
* and the endpoint slope ($k_4$),

producing a highly accurate estimate of the **average slope over the interval**.

It can be viewed as constructing a **curved path approximation** rather than a straight tangent step.

---

## Stability

Apply to the test equation:

$$
y' = \lambda y
$$

The method produces:

$$
y_{k+1} = R(h\lambda),y_k
$$

where:

$$
R(z) = 1 + z + \frac{z^2}{2} + \frac{z^3}{6} + \frac{z^4}{24}
$$

This is the fourth-order Taylor polynomial of $e^z$.

Stability requires:

$$
|R(z)| < 1
$$

The stability region is significantly larger than Euler’s, but still bounded, so RK4 is **not A-stable**.

---

## RK4 as a Quadrature Rule

Recall:

$$
y(T) = y_0 + \int_{t_0}^T f(t, y(t)),dt
$$

RK4 corresponds to a **Simpson-like quadrature along the solution path**, using carefully chosen intermediate evaluations.

---

## Summary

| Property             | Value                                      |
| -------------------- | ------------------------------------------ |
| Method type          | Explicit Runge–Kutta (4-stage)             |
| Order                | 4                                          |
| Function evaluations | 4 per step                                 |
| LTE                  | $O(h^5)$                                   |
| Global error         | $O(h^4)$                                   |
| Stability            | Much improved over Euler, but not A-stable |


In [3]:
def runge_kutta_ode(func, y0, t0, tf, dt):
    """
    Solves an ordinary differential equation (ODE) using the Runge-Kutta method (4th order).

    Parameters:
    func: A function that takes two arguments (t, y) and returns the derivative dy/dt.
    y0: The initial value of y at time t0.
    t0: The initial time.
    tf: The final time.
    dt: The time step size.

    Returns:
    A list of tuples (t, y) representing the time and corresponding value of y at each time step.
    """
    times = []
    values = []
    
    t = t0
    y = y0
    
    while t <= tf:
        times.append(t)
        values.append(y)
        
        # Compute the four slopes
        k1 = func(t, y)
        k2 = func(t + dt / 2, y + k1 * dt / 2)
        k3 = func(t + dt / 2, y + k2 * dt / 2)
        k4 = func(t + dt, y + k3 * dt)
        
        # Update y using the weighted average of the slopes
        y += (k1 + 2 * k2 + 2 * k3 + k4) * dt / 6
        
        # Increment time by dt
        t += dt
    
    return list(zip(times, values))

# General Runge–Kutta Methods

## The Problem

We seek a general framework for constructing numerical methods for:

$$
\frac{dy}{dt} = f(t, y), \qquad y(t_0) = y_0
$$

that achieve high-order accuracy using only evaluations of $f$.

---

## The General RK Scheme

An $s$-stage Runge–Kutta method is defined by:

$$
k_i = f\left(t_k + c_i h,; y_k + h \sum_{j=1}^{s} a_{ij} k_j\right), \quad i = 1, \dots, s
$$

$$
y_{k+1} = y_k + h \sum_{i=1}^{s} b_i k_i
$$

---

## Structure

* $k_i$: stage slopes
* $a_{ij}$: stage coupling coefficients
* $b_i$: weights in the final update
* $c_i$: time nodes

For **explicit methods**:

$$
a_{ij} = 0 \quad \text{for } j \ge i
$$

---

## Butcher Tableau (Readable Form)


| $c_i$    | $a_{i1}$ | $a_{i2}$ | $\cdots$ | $a_{is}$ |
| -------- | -------- | -------- | -------- | -------- |
| $c_1$    | 0        |          |          |          |
| $c_2$    | $a_{21}$ | 0        |          |          |
| $\vdots$ | $\vdots$ | $\vdots$ | $\ddots$ |          |
| $c_s$    | $a_{s1}$ | $a_{s2}$ | $\cdots$ | 0        |

Final weights:

$$
(b_1, b_2, \dots, b_s)
$$

---

## Order Conditions (Idea)

To achieve order $p$, match the Taylor expansion:

$$
y(t_k + h) = y(t_k) + h y' + \frac{h^2}{2}y'' + \cdots + \frac{h^p}{p!}y^{(p)} + O(h^{p+1})
$$

This yields algebraic constraints on $(a_{ij}, b_i, c_i)$.

---

### First Conditions

Consistency:

$$
\sum_{i=1}^s b_i = 1
$$

Second order:

$$
\sum_{i=1}^s b_i c_i = \frac{1}{2}
$$

---

## Examples

### Euler (RK1)

| $c$ | $a$ |
| --- | --- |
| 0   | 0   |

$$
b = (1)
$$

---

### Heun (RK2)

| $c$ | $a_1$ | $a_2$ |
| --- | ----- | ----- |
| 0   | 0     | 0     |
| 1   | 1     | 0     |

$$
b = \left(\frac{1}{2}, \frac{1}{2}\right)
$$

---

### RK4

| $c$            | $a_1$          | $a_2$          | $a_3$ | $a_4$ |
| -------------- | -------------- | -------------- | ----- | ----- |
| 0              | 0              | 0              | 0     | 0     |
| $\tfrac{1}{2}$ | $\tfrac{1}{2}$ | 0              | 0     | 0     |
| $\tfrac{1}{2}$ | 0              | $\tfrac{1}{2}$ | 0     | 0     |
| 1              | 0              | 0              | 1     | 0     |

$$
b = \left(\frac{1}{6}, \frac{1}{3}, \frac{1}{3}, \frac{1}{6}\right)
$$

---

## Stability Function

Applying RK to:

$$
y' = \lambda y
$$

gives:

$$
y_{k+1} = R(h\lambda),y_k
$$

Stability requires:

$$
|R(z)| < 1
$$

---

## Interpretation

Runge–Kutta methods approximate:

$$
y(t_{k+1}) = y(t_k) + \int_{t_k}^{t_{k+1}} f(t, y(t)),dt
$$

by constructing a high-order quadrature rule along the solution curve using internal stage evaluations.

---

## Summary

| Property     | Description                    |
| ------------ | ------------------------------ |
| General form | Weighted sum of stage slopes   |
| Explicit RK  | No solves required             |
| Implicit RK  | Requires solving systems       |
| Accuracy     | Determined by order conditions |


# Adaptive Runge–Kutta Methods

## Background: Fixed-Step RK4

The classical fourth-order Runge–Kutta method advances the IVP $y' = f(t,y)$, $y(t_0) = y_0$
by computing four slope estimates per step:

$$k_1 = f(t_k,\, y_k)$$
$$k_2 = f\!\left(t_k + \tfrac{h}{2},\, y_k + \tfrac{h}{2}k_1\right)$$
$$k_3 = f\!\left(t_k + \tfrac{h}{2},\, y_k + \tfrac{h}{2}k_2\right)$$
$$k_4 = f(t_k + h,\, y_k + hk_3)$$

$$y_{k+1} = y_k + \frac{h}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$

This is $O(h^4)$ globally — but with a **fixed** $h$ chosen blindly for the whole interval.
Like fixed-grid quadrature, this wastes evaluations in smooth regions and can be inaccurate
through sharp features. **Adaptive RK** fixes this by letting $h$ vary automatically.

---

## The Core Idea: Embedded Pairs

An **embedded RK pair** computes two approximations of different orders
using the **same set of stage evaluations** $k_1, \dots, k_s$:

$$y_{k+1}   = y_k + h\sum_{i=1}^s b_i\, k_i \quad \text{(order } p\text{)}$$
$$\hat{y}_{k+1} = y_k + h\sum_{i=1}^s \hat{b}_i\, k_i \quad \text{(order } p+1\text{)}$$

Their difference is a **local error estimate** at essentially zero extra cost:

$$\mathbf{e}_{k+1} = \hat{y}_{k+1} - y_{k+1} = h\sum_{i=1}^s (\hat{b}_i - b_i)\,k_i$$

This is the key idea: one set of $f$-evaluations, two accuracy levels, free error control.

---

## The Dormand–Prince Method (RK45 / `dopri5`)

The most widely used embedded pair is **Dormand–Prince**, which computes
six stage values (with a 7th reused from the next step):

$$k_1 = f(t_k,\; y_k)$$
$$k_2 = f\!\left(t_k + \tfrac{1}{5}h,\; y_k + \tfrac{h}{5}k_1\right)$$
$$k_3 = f\!\left(t_k + \tfrac{3}{10}h,\; y_k + h\left[\tfrac{3}{40}k_1 + \tfrac{9}{40}k_2\right]\right)$$
$$k_4 = f\!\left(t_k + \tfrac{4}{5}h,\; y_k + h\left[\tfrac{44}{45}k_1 - \tfrac{56}{15}k_2 + \tfrac{32}{9}k_3\right]\right)$$
$$k_5 = f\!\left(t_k + \tfrac{8}{9}h,\; y_k + h\left[\tfrac{19372}{6561}k_1 - \tfrac{25360}{2187}k_2 + \tfrac{64448}{6561}k_3 - \tfrac{212}{729}k_4\right]\right)$$
$$k_6 = f\!\left(t_k + h,\; y_k + h\left[\tfrac{9017}{3168}k_1 - \tfrac{355}{33}k_2 + \tfrac{46732}{5247}k_3 + \tfrac{49}{176}k_4 - \tfrac{5103}{18656}k_5\right]\right)$$

The **4th-order** solution (used to advance):

$$y_{k+1} = y_k + h\left[\frac{35}{384}k_1 + \frac{500}{1113}k_3 + \frac{125}{192}k_4 - \frac{2187}{6784}k_5 + \frac{11}{84}k_6\right]$$

The **5th-order** solution (used only for error estimation):

$$\hat{y}_{k+1} = y_k + h\left[\frac{5179}{57600}k_1 + \frac{7571}{16695}k_3 + \frac{393}{640}k_4 - \frac{92097}{339200}k_5 + \frac{187}{2100}k_6 + \frac{1}{40}k_7\right]$$

where $k_7 = f(t_{k+1}, y_{k+1})$ — which is also $k_1$ of the **next** step.

---

## FSAL: First Same As Last

The genius of Dormand–Prince is that $k_7 = f(t_{k+1}, y_{k+1})$ becomes $k_1$ of the following step.
This **First Same As Last** (FSAL) trick means each accepted step costs effectively **5 new
$f$-evaluations** rather than 6, since the last stage is recycled. Only rejected steps waste an evaluation.

---

## Step-Size Control

After each attempted step, compute a scalar error measure. A standard mixed absolute/relative norm:

$$\text{err} = \left\|\frac{\mathbf{e}_{k+1}}{\text{atol} + \text{rtol}\cdot\max(|y_k|, |y_{k+1}|)}\right\|_{\text{RMS}}$$

where $\text{atol}$ and $\text{rtol}$ are user-specified absolute and relative tolerances.

**Accept** the step if $\text{err} \leq 1$ and advance $t_{k+1} = t_k + h$.

**Reject** the step if $\text{err} > 1$, discard $y_{k+1}$, and retry with a smaller $h$.

In both cases, propose a new step size using the **optimal scaling formula**:

$$\boxed{h_{\text{new}} = h \cdot \min\!\left(h_{\max},\, \max\!\left(h_{\min},\, S \cdot \left(\frac{1}{\text{err}}\right)^{1/(p+1)}\right)\right)}$$

where $p = 4$ is the order of the advancing formula, $S \approx 0.9$ is a **safety factor**,
and $h_{\min}$, $h_{\max}$ are step-size bounds.

---

## Where Does the Exponent $1/(p+1)$ Come From?

The local error of a $p$-th order method scales as:

$$\|\mathbf{e}\| \approx C h^{p+1}$$

for some constant $C$ that depends on higher derivatives of $f$. If the current error estimate is
$\text{err} \approx C h^{p+1} / \text{tol}$, we want a new $h^*$ such that $C (h^*)^{p+1} / \text{tol} = 1$:

$$h^* = h \cdot \left(\frac{\text{tol}}{C h^{p+1}}\right)^{1/(p+1)} = h \cdot \left(\frac{1}{\text{err}}\right)^{1/(p+1)}$$

The safety factor $S < 1$ makes the target slightly below tolerance to reduce the chance of the
very next step also being rejected.

---

## The Full Adaptive Algorithm

**Initialize:** $t = t_0$, $y = y_0$, $h = h_{\text{init}}$.

**Loop** until $t \geq T$:

1. Compute $k_1, \dots, k_6$ and form $y_{k+1}$, $\hat{y}_{k+1}$.
2. Compute $\mathbf{e} = \hat{y}_{k+1} - y_{k+1}$ and $\text{err}$.
3. If $\text{err} \leq 1$:
   - **Accept**: set $t \leftarrow t + h$, $y \leftarrow y_{k+1}$.
   - Save $k_7 = f(t, y)$ as $k_1$ for next step (FSAL).
4. Compute $h_{\text{new}}$ from the scaling formula.
5. Set $h \leftarrow \min(h_{\text{new}},\, T - t)$ to hit $T$ exactly.

---

## Error and Convergence

For a smooth IVP, the Dormand–Prince method achieves:

$$\|y(T) - y_N\| \leq C \cdot \text{tol} \cdot (T - t_0) + O(\text{tol}^{5/4})$$

The dominant term is proportional to the user tolerance — **the global error tracks the
tolerance** automatically, unlike fixed-step methods where the relationship between $h$ and
global error must be determined manually.

Work (number of $f$-evaluations) scales as:

$$W \sim \frac{T - t_0}{h_{\text{avg}}} \cdot 5 \approx 5(T-t_0) \cdot C^{1/5} \cdot \text{tol}^{-1/5}$$

The $\text{tol}^{-1/5}$ scaling means **halving the tolerance costs only $2^{1/5} \approx 1.15\times$ more work** — remarkably cheap for a full order-of-magnitude gain in accuracy.

---

## Other Common Embedded Pairs

| Name | Orders | Stages | Notes |
|---|---|---|---|
| Fehlberg RK45 (RKF45) | 4/5 | 6 | Original embedded pair; less accurate than DOPRI |
| **Dormand–Prince (DOPRI5)** | **4/5** | **6+FSAL** | **Default in `scipy`, MATLAB `ode45`** |
| Bogacki–Shampine (BS23) | 2/3 | 3+FSAL | Default `ode23`; cheap for loose tolerances |
| Cash–Karp | 4/5 | 6 | Popular in physics; good for oscillatory problems |
| Verner (DVERK) | 6/7 | 8 | Higher accuracy per step; fewer steps for tight tol |
| Dormand–Prince 8/5/3 | 7/8 | 13 | Used in `DOP853`; tight-tolerance workhorse |

---

## Stiff Problems and Implicit Methods

Adaptive RK methods like DOPRI5 are **explicit** — each $k_i$ is computed directly from
known values. For **stiff** problems (where $\lambda_{\max} \gg \lambda_{\min}$ in the Jacobian),
the step size is controlled by stability rather than accuracy, forcing tiny $h$ regardless of smoothness.

The remedy is **implicit** embedded pairs:

- **Rosenbrock methods**: linearise $f$ once per step, solve a linear system — good for mildly stiff problems.
- **SDIRK pairs** (Singly Diagonally Implicit RK): implicit stages with the same diagonal coefficient — more robust, used in `ode23s`.
- **Radau IIA**: fully implicit, $A$-stable, $O(h^{2s-1})$ order — the gold standard for stiff problems (`scipy.integrate.solve_ivp` with `method='Radau'`).

---

## Connection to Adaptive Quadrature

The structure is identical to adaptive quadrature from the previous note:

| Adaptive quadrature | Adaptive RK |
|---|---|
| Two rules of different order | Embedded RK pair of orders $p$ and $p+1$ |
| $\|Q_2 - Q_1\|$ as error estimate | $\|\hat{y} - y\|$ as error estimate |
| Bisect the interval | Shrink the step size $h$ |
| Accept panel if error $< \varepsilon$ | Accept step if err $\leq 1$ |
| Global error $\leq \varepsilon$ | Global error $\propto$ tol |

The only structural difference is that ODE errors compound through the dynamics $f$
(via the Lipschitz constant), while quadrature errors simply sum — which is why the ODE
error bound carries an $e^{L(T-t_0)}$ factor.

---

## Conditions and Caveats

| Concern | Detail |
|---|---|
| Stiffness | Explicit adaptive RK fails — switch to implicit method |
| Discontinuous $f$ | Step-size controller stalls near jump; locate and bracket discontinuities |
| Dense output | Standard adaptive RK gives values only at accepted nodes; use **continuous extension** polynomials for values at arbitrary $t$ |
| Event detection | Roots of $g(t, y(t)) = 0$ (e.g. crossing a threshold) require bisection within a step |
| Very tight tolerances | Switch to higher-order pair (DOP853) to avoid excessive rejected steps |

# Shooting Method for Boundary Value Problems

## Initial Value vs Boundary Value Problems

### Initial Value Problem (IVP)

An IVP specifies all conditions at a single point:

$$
\frac{dy}{dt} = f(t, y), \qquad y(t_0) = y_0
$$

The solution is obtained by propagating forward from $t_0$.

---

### Boundary Value Problem (BVP)

A BVP specifies conditions at multiple points:

$$
y'' = g(t, y, y'), \qquad y(a) = \alpha, \quad y(b) = \beta
$$

The solution must satisfy constraints at both ends of the interval.

---

## The Key Idea: Convert BVP → IVP

For a second-order equation:

$$
y'' = g(t, y, y')
$$

introduce:

$$
y_1 = y
$$

$$
y_2 = y'
$$

giving the system:

$$
y_1' = y_2
$$

$$
y_2' = g(t, y_1, y_2)
$$

We know:

$$
y_1(a) = \alpha
$$

but the initial slope:

$$
y_2(a) = y'(a)
$$

is unknown.

---

## The Shooting Idea

Guess the missing slope:

$$
y'(a) = s
$$

Solve the IVP:

$$
y_1' = y_2
$$

$$
y_2' = g(t, y_1, y_2)
$$

$$
y_1(a) = \alpha
$$

$$
y_2(a) = s
$$

This produces a solution $y(t; s)$.

---

## Root-Finding Formulation

Define:

$$
\phi(s) = y(b; s) - \beta
$$

We seek:

$$
\phi(s) = 0
$$

---

## The Algorithm

1. Choose an initial guess $s_0$
2. Solve the IVP
3. Compute:
   $$
   \phi(s_0) = y(b; s_0) - \beta
   $$
4. Update $s$ (secant or Newton)
5. Repeat until convergence

---

## Secant Shooting Method

Given $s_0, s_1$:

$$
s_{k+1}
= s_k - \phi(s_k)\frac{s_k - s_{k-1}}{\phi(s_k) - \phi(s_{k-1})}
$$

---

## Interpretation

Each guess $s$ produces a trajectory starting at $t=a$.
We adjust $s$ until the solution satisfies:

$$
y(b) = \beta
$$

---

## Summary

| Concept   | Idea                        |
| --------- | --------------------------- |
| BVP issue | Unknown initial slope       |
| Method    | Guess → solve IVP → correct |
| Reduction | Root-finding problem        |
| Function  | $\phi(s) = y(b; s) - \beta$ |


In [4]:
def shooting_method(func, y0, t0, tf, dt, target):
    """
    Solves a boundary value problem (BVP) using the shooting method.

    Parameters:
    func: A function that takes two arguments (t, y) and returns the derivative dy/dt.
    y0: The initial guess for the value of y at time t0.
    t0: The initial time.
    tf: The final time.
    dt: The time step size.
    target: The target value of y at time tf.

    Returns:
    A list of tuples (t, y) representing the time and corresponding value of y at each time step.
    """
    def objective(y_guess):
        # Solve the ODE with the given guess for the initial condition
        solution = euler_method_ode(func, y_guess, t0, tf, dt)
        # Return the difference between the final value and the target
        return solution[-1][1] - target
    
    # Use a root-finding method to find the correct initial condition
    from scipy.optimize import root_scalar
    result = root_scalar(objective, bracket=[y0 - 10, y0 + 10], method='bisect')
    
    if not result.converged:
        raise ValueError("Shooting method did not converge.")
    
    # Solve the ODE with the found initial condition
    return euler_method_ode(func, result.root, t0, tf, dt)